In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigs
from scipy import linalg
from sklearn.metrics.pairwise import cosine_similarity

In [97]:
def calculate_ipr(vector):
    """
    Calculate the Inverse Participation Ratio (IPR) for a given vector.
    IPR is the sum of the fourth power of the elements in the vector.
    """
    return np.sum(vector**4)

def update_regularization_matrix(Z, A, feature_matrix, threshold=0.1):
    """
    Update the diagonal elements of matrix Z based on the IPR of the feature vectors.
    A penalty is applied if the IPR exceeds a certain threshold (indicating high localization).
    
    Parameters:
    - Z: Initial regularization matrix (diagonal, 4x4 in this case for 4 nodes).
    - feature_matrix: The matrix where each row is a feature vector (10x4).
    - threshold: A value to decide when to apply a penalty based on IPR.
    """
    for i, vector in enumerate(feature_matrix):  # Iterate over the columns (nodes' feature vectors)
        ipr_value = calculate_ipr(vector)
        if ipr_value > threshold:
            # Apply a penalty by modifying the diagonal element of Z
            Z[i, i] += 1  # Example penalty: Add the IPR to the diagonal
    return Z

# def compute_z_laplacian(A, feature_matrix, num_components=9, penalty_factor=0.5):
#     """
#     Computes the Z-Laplacian matrix from the adjacency matrix A and feature matrix, applying penalties
#     based on eigenvalue localization and feature similarity.

#     Parameters:
#     A : np.ndarray
#         Adjacency matrix of the graph (n x n)
#     feature_matrix : np.ndarray
#         Feature matrix (n x d), where n is the number of nodes and d is the feature dimension.
#     num_components : int
#         Number of primary components (largest eigenvalues) to apply stronger penalties to.
#     penalty_factor : float
#         Factor by which to penalize the largest eigenvalues (0 < penalty_factor < 1).

#     Returns:
#     Lz : np.ndarray
#         The Z-Laplacian matrix (n x n).
#     """
    
#     # Step 1: Eigen decomposition of adjacency matrix A
#     eigenvalues, eigenvectors = eigh(A)
#     num_components = A.shape[0]
    
#     # Step 2: Calculate the feature similarity matrix (cosine similarity)
#     # This matrix represents how similar the features are between different nodes
#     feature_similarity = cosine_similarity(feature_matrix)
    
#     # Normalize the feature similarity matrix to have values between 0 and 1
#     feature_similarity = (feature_similarity - feature_similarity.min()) / (feature_similarity.max() - feature_similarity.min())
    
#     # Step 3: Penalize the eigenvalues based on the feature similarity
#     regularized_eigenvalues = eigenvalues.copy()
    
#     # Use the feature similarity to penalize eigenvalues differently for each component
#     for i in range(num_components):
#         # Compute an average penalty factor based on feature similarity of the nodes
#         penalty = penalty_factor * np.mean(feature_similarity[i])
#         regularized_eigenvalues[i] *= penalty  # Apply penalty to the top i largest eigenvalues
    
#     # Step 4: Construct the Z matrix as a diagonal matrix using regularized eigenvalues
#     Z = np.diag(regularized_eigenvalues)
    
#     # Step 5: Compute the Z-Laplacian as Lz = A + Z
#     Lz = A + Z
    
#     return Lz

In [114]:
# Function to initialize P randomly
def initialize_P(n, k):
    P = np.random.rand(n, k)
    P /= np.sum(P, axis=1)[:, np.newaxis]
    return P

# Function to compute the regularized Laplacian (Lz)
def compute_regularized_laplacian(A, Z):
    return A + Z

# Non-backtracking matrix-based method to estimate number of communities
def estimate_num_communities(A, features, initial_k=4, step=1, max_k=10):
    # Step 1: Calculate the degrees of the nodes
    degrees = np.sum(A, axis=1)
    diag_degrees = diags(degrees)
    
    # Step 2: Construct the non-backtracking matrix
    non_backtracking_matrix = np.block([
        [A, diag_degrees - np.eye(len(A))],
        [-np.eye(len(A)), np.zeros_like(A)]
    ])
    
    # Step 3: Compute the norm of the non-backtracking matrix
    radius = np.sqrt(np.linalg.norm(non_backtracking_matrix,'fro'))  # Frobenius norm
    print("Radius of the non-backtracking matrix:", radius)
    # radius = np.sqrt(np.sum(np.abs(non_backtracking_matrix.data)**2))

    k = initial_k
    communities_count = 0
    all_real_found = False

    eigvals = linalg.eigvals(non_backtracking_matrix)
    # eigvals = np.sort(eigvals)[::-1]
    # plot the eigenvalues
    plt.plot(eigvals.real, eigvals.imag, 'o')
    plt.xlabel('Re')
    plt.ylabel('Im')
    plt.title('Eigenvalues of the non-backtracking matrix')
    plt.show()
    
    # print the eigenvalues in a file
    np.savetxt('eigenvalues.txt', eigvals)
    while not all_real_found and k <= max_k:
    # while k <= max_k:
        # Step 4: Compute the eigenvalues of the non-backtracking matrix
        # sort the eigenvalues in descending order
        # get the top k eigenvalues
        k_eigvals = eigvals[:k]
        print(k_eigvals)
        # print(radius)
        
        # Step 5: Filter out eigenvalues inside the circle with radius equal to the norm of the matrix
        eigen_distances = np.abs(k_eigvals)  # Magnitude of each eigenvalue
        outside_circle = eigen_distances > radius
        
        # Count the number of eigenvalues outside the circle
        communities_count = np.sum(outside_circle)
        print(f"Number of eigenvalues outside the circle: {communities_count}")
        if communities_count > 0:
           all_real_found = True  # Stop if we have enough eigenvalues to count
        
        else:
            # Increase k and try again
            k += step

    return communities_count

# Symmetric NMF update rule
def update_P(Lz, P, beta=0.5):
    numerator = np.dot(Lz, P)
    denominator = np.dot(np.dot(P, P.T), P)
    denominator[denominator == 0] = 1e-10  # Avoid division by zero
    P = P * ((1 - beta) + beta * numerator / denominator)
    P_updated = np.maximum(np.dot(np.dot(Lz, P), np.linalg.inv(np.dot(P.T, P))), 0) # Non-negative constraint
    return P_updated

# CDNMF algorithm
def CDNMF(A, features, num_iterations=100, beta=0.5):
    n = A.shape[0]
    
    # Step 1: Estimate the number of communities (k) using spectral analysis
    # k = estimate_num_communities(A, features)
    k = 7  # For simplicity, we will use a fixed number of communities
    print(f"Estimated number of communities: {k}")
    
    # Step 2: Initialize P matrix randomly
    P = initialize_P(n, k)
    
    # Step 3: Initialize regularization matrix Z
    Z = np.zeros((n, n))
    Z = update_regularization_matrix(Z, A, features, threshold=0.5)
    
    # Step 4: Compute regularized Laplacian Lz
    Lz = A + Z
    
    # Step 5: Iteratively update P using the update rule
    obj_value = np.inf
    for iteration in range(num_iterations):
        P = update_P(Lz, P, beta)

        # Compute the objective function value
        obj_value = np.linalg.norm(A - np.dot(P, P.T), 'fro')
        if iteration % 10 == 0:
            print(f"Objective function value after {iteration+1} iterations: {obj_value}")

        # If the objective value is below a certain threshold, stop the iterations
        if obj_value < 1e-6:
            break

    # Normalize P so that each row sums to 1
    P /= np.sum(P, axis=1)[:, np.newaxis]
    
    # Assign nodes to communities based on maximum membership value
    communities = np.argmax(P, axis=1)
    
    return communities, P


In [127]:
# Function to parse .cites and .content files and build the adjacency matrix
def parse_dataset(cites_file, content_file):
    # Parse the .content file
    features = []
    node_id_to_index = {}
    class_labels = set()
    actual_labels = []
    num_nodes = 0
    with open(content_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            node_id = parts[0]
            node_id_to_index[node_id] = num_nodes

            feature_vector = np.array([float(x) for x in parts[1:-1]])
            feature_vector = feature_vector / np.linalg.norm(feature_vector)
            features.append(feature_vector)

            class_label = parts[-1]
            class_labels.add(class_label)
            actual_labels.append(class_label)
            
            num_nodes += 1

    # make number of features consistent
    max_features = max([len(feature) for feature in features])
    for i in range(len(features)):
        features[i] = np.pad(features[i], (0, max_features - len(features[i])), 'constant')
    
    features = np.array(features)

    # Parse the .cites file
    edges = []
    with open(cites_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            edges.append((parts[0], parts[1]))
    
    print(f"Number of nodes: {num_nodes}")
    print(f"Number of edges: {len(edges)}")
    print(f"Feature dimension: {len(features[0])}")
    print(f"Number of unique class labels: {len(class_labels)}")
    
    # Build the adjacency matrix
    A = np.zeros((num_nodes, num_nodes))
    for edge in edges:
        if edge[0] not in node_id_to_index or edge[1] not in node_id_to_index:
            continue
        x = node_id_to_index[edge[0]]
        y = node_id_to_index[edge[1]]
        A[x, y] = 1
        A[y, x] = 1
        # print(f"Adding edge between {x} and {y}")

    # # Convert directed graph to undirected graph by duplicating the nodes
    # A = np.zeros((2*num_nodes, 2*num_nodes))
    # for edge in edges:
    #     if edge[0] not in node_id_to_index or edge[1] not in node_id_to_index:
    #         continue
    #     x = node_id_to_index[edge[0]]
    #     y = node_id_to_index[edge[1]]
    #     A[x, y] = 1
    #     A[y, x] = 1
    #     A[x, x+num_nodes] = 1
    #     A[x+num_nodes, x] = 1
    #     A[y, y+num_nodes] = 1
    #     A[y+num_nodes, y] = 1
    #     A[x+num_nodes, y+num_nodes] = 1
    #     A[y+num_nodes, x+num_nodes] = 1
    #     A[x, y+num_nodes] = 1
    #     A[y+num_nodes, x] = 1

    # features = np.concatenate((features, features), axis=0)
    # actual_labels = actual_labels + actual_labels
    
    return A, features, actual_labels

In [128]:
def calculate_modularity(A, communities):
    """
    Calculate modularity of a graph with adjacency matrix A and community assignments.
    
    A: numpy array (adjacency matrix)
    communities: list or array of community assignments for each node
    """
    num_nodes = A.shape[0]
    m = np.sum(A)  # Total number of edges in the graph
    modularity = 0.0
    
    # Calculate degree of each node
    degrees = np.sum(A, axis=1)
    
    for i in range(num_nodes):
        for j in range(num_nodes):
            if communities[i] == communities[j]:  # Only consider nodes in the same community
                modularity += A[i, j] - (degrees[i] * degrees[j]) / (2 * m)
    
    return modularity / (2 * m)

In [129]:
# Example usage
if __name__ == "__main__":
    # Load the dataset
    cites_file = "Datasets/cora/cora.cites"
    content_file = "Datasets/cora/cora.content"
    # cites_file = "Datasets/citeseer-doc-classification/citeseer.cites"
    # content_file = "Datasets/citeseer-doc-classification/citeseer.content"
    A, features, actual_labels = parse_dataset(cites_file, content_file)

    # Run the CDNMF algorithm
    communities, P = CDNMF(A, features, num_iterations=100, beta=0.5)
    
    # Output community assignment
    print("Community assignment:", communities)
    # print("Community membership matrix P:\n", P)

    # Calculate modularity
    calc_modularity = calculate_modularity(A, communities)
    orig_modularity = calculate_modularity(A, actual_labels)
    print(f"Original modularity: {orig_modularity}")
    print(f"Calculated modularity: {calc_modularity}")

Number of nodes: 2708
Number of edges: 5429
Feature dimension: 1433
Number of unique class labels: 7
Estimated number of communities: 7
Objective function value after 1 iterations: 102.62896339895504
Objective function value after 11 iterations: 99.36460518478466
Objective function value after 21 iterations: 99.34727244948824
Objective function value after 31 iterations: 99.34561858453522
Objective function value after 41 iterations: 99.34512837811299
Objective function value after 51 iterations: 99.34481520466002
Objective function value after 61 iterations: 99.34444609151764
Objective function value after 71 iterations: 99.34394367832873
Objective function value after 81 iterations: 99.34329234701404
Objective function value after 91 iterations: 99.34260021098666
Community assignment: [4 3 1 ... 0 1 4]
Original modularity: 0.3625211766867525
Calculated modularity: 0.38374458902680203
